In [3]:
import os
from getpass import getpass

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool
import yfinance as yf
from duckduckgo_search import DDGS
import agentops
import sys

os.environ["PYTHONIOENCODING"] = "utf-8"

# Load environment variables from the local .env file if present.
load_dotenv()

# Configure the Gemini API key. Prompt the user if it is not set in the environment.
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    api_key = getpass("Enter your Gemini API key: ")
os.environ["GEMINI_API_KEY"] = api_key

# Load AgentOps API key if not already set.
agentops_api_key = os.getenv("AGENTOPS_API_KEY")
if not agentops_api_key:
    agentops_api_key = getpass("Enter your AgentOps API key: ")
os.environ["AGENTOPS_API_KEY"] = agentops_api_key

# Initialize AgentOps.
agentops.init(api_key=os.environ["AGENTOPS_API_KEY"])

# Initialize the CrewAI LLM client for Gemini.
llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=api_key,
    temperature=0.7,
)


class StockSearchTool(BaseTool):
    name: str = "StockNewsSearcher"
    description: str = "Search for the latest news and updates about a stock using DuckDuckGo"

    def _run(self, query: str) -> str:
        # Use DuckDuckGo to gather a compact set of recent news results for the requested stock.
        try:
            with DDGS() as ddgs:
                results = ddgs.text(query, max_results=2)
            if not results:
                return "No news results were found."
            return "\n".join([r.get("body", "") for r in results if r.get("body")])
        except Exception as exc:
            return f"News search failed: {exc}"


class YahooFinanceTool(BaseTool):
    name: str = "YahooFinanceFetcher"
    description: str = "Get the latest 1-month stock price history for a given ticker using yFinance"

    def _run(self, ticker: str) -> str:
        # Fetch the most recent price history to support the price analysis task.
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period="1mo")
            if hist.empty:
                return f"No price data available for {ticker}."
            return hist.tail(3).to_string()
        except Exception as exc:
            return f"Price data lookup failed: {exc}"


# Define the two specialized agents for the multi-agent workflow.
stock_analyst = Agent(
    role="Stock Analyst",
    goal="Analyze recent stock data and news",
    backstory="Expert in financial trends, macro indicators, and company performance",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    agentops_enabled=True,
)

report_writer = Agent(
    role="Report Generator",
    goal="Write investor-friendly summaries of stock analysis",
    backstory="Professional writer with expertise in finance reporting",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    agentops_enabled=True,
)

# Create tasks for news gathering, trend analysis, and report generation.
search_tool = StockSearchTool()
finance_tool = YahooFinanceTool()

search_task = Task(
    description="Search latest news and updates about the stock 'AAPL' using DuckDuckGo.",
    expected_output="Summarized news highlights for Apple stock.",
    agent=stock_analyst,
    tools=[search_tool],
)

analysis_task = Task(
    description="Analyze Apple stock price trends using yFinance.",
    expected_output="Key trends and technical highlights for the past month.",
    agent=stock_analyst,
    tools=[finance_tool],
)

report_task = Task(
    description="Write a clean investor report using previous analysis and news insights.",
    expected_output="Concise report with market summary and investment outlook.",
    agent=report_writer,
)

# Assemble the crew so the agents can work together in sequence.
crew = Crew(
    agents=[stock_analyst, report_writer],
    tasks=[search_task, analysis_task, report_task],
    verbose=False,
)

In [4]:
# Run the agent crew and print the final investor report.
result = await crew.kickoff_async()

print("\n📊 Final Stock Analysis Report:\n")
print(result)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Task: Search latest news and updates about the stock 'AAPL' using DuckDuckGo.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

C:\Users\Shubham\AppData\Local\Temp\ipykernel_16792\368469182.py:47: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Tool stock_news_searcher executed with result: Find the latest Apple Inc. (AAPL) stock quote, history, news and other vital information to help you with your stock trading and investing.
Apple Inc. is an American multinational technology company h...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Find the latest Apple Inc. (AAPL) stock quote, history, news and other vital information to help you with      │
│  your stock trading and investing.                                                                              │
│  Apple Inc. is an American multinational technology company headquartered in Cupertino, California, in Silicon  │
│  Valley, and known for …                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1619: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1619: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1626: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1627: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


...

c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1619: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1619: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1626: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1627: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Generator                                                                                        │
│                                                                                                                 │
│  Task: Write a clean investor report using previous analysis and news insights.                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Generator                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Apple Inc. (AAPL) Investor Report**                                                                          │
│                                                                                                                 │
│  **Date:** [Current Date]                                                                                       │
│                                                                                                                 │
│  **Company Overview:**                                                                                          │
│  Apple Inc. (AAPL) is an American multinational technology company headquartered in Cupertino, California,      │
│  renowned for its innovative consumer electronics, software, and online services. The company's diverse         │
│  product ecosystem, including iPhones, Macs, iPads, Apple Watch, and services like the App Store and Apple      │
│  Music, underpins its strong market position and brand loyalty.                                                 │
│                                                                                                                 │
│  **Market Summary (July 15-17, 2026 - Limited Data Analysis):**                                                 │
│  *Please note: The analysis below is based on a highly limited, three-day dataset from July 15-17, 2026, which  │
│  appears to be future-dated. This short timeframe does not allow for a comprehensive assessment of long-term    │
│  or even typical short-term market trends for Apple Inc.*                                                       │
│                                                                                                                 │
│  During the observed three-day period from July 15th to July 17th, 2026, Apple Inc. (AAPL) exhibited a          │
│  distinct upward price trajectory accompanied by increasing trading volume.                                     │
│                                                                                                                 │
│  *   **Price Performance:** AAPL opened at $317.62 on July 15th and closed at $333.74 on July 17th, marking a   │
│  significant gain within this brief window. The stock consistently achieved higher daily highs and higher       │
│  daily lows, indicating strong buying momentum.                                                                 │
│  *   **Volume Analysis:** Trading volume steadily increased each day, from 60,957,600 shares on July 15th to    │
│  63,365,300 shares on July 17th. This rising volume alongside increasing prices typically suggests growing      │
│  investor interest and conviction in the stock's upward movement during this specific period.                   │
│  *   **Technical Momentum:** The consistent upward movement, coupled with increasing volume, points to          │
│  positive short-term technical momentum within this very constrained dataset.                                   │
│                                                                                                                 │
│  **Investment Outlook:**                                                                                        │
│  Given the extremely limited and future-dated nature of the provided price data, it is not possible to          │
│  formulate a robust or reliable long-term investment ou


📊 Final Stock Analysis Report:

**Apple Inc. (AAPL) Investor Report**

**Date:** [Current Date]

**Company Overview:**
Apple Inc. (AAPL) is an American multinational technology company headquartered in Cupertino, California, renowned for its innovative consumer electronics, software, and online services. The company's diverse product ecosystem, including iPhones, Macs, iPads, Apple Watch, and services like the App Store and Apple Music, underpins its strong market position and brand loyalty.

**Market Summary (July 15-17, 2026 - Limited Data Analysis):**
*Please note: The analysis below is based on a highly limited, three-day dataset from July 15-17, 2026, which appears to be future-dated. This short timeframe does not allow for a comprehensive assessment of long-term or even typical short-term market trends for Apple Inc.*

During the observed three-day period from July 15th to July 17th, 2026, Apple Inc. (AAPL) exhibited a distinct upward price trajectory accompanied by increasing t